# Faruq-v3 — retained models on synthetic density
Evaluasi **tanpa training** seluruh kandidat RETAIN/PASS seed-42 pada B0–B3 sintetis yang sudah tersedia. GEO1 masuk; GEO-SHARED60/FAM35x3 belum masuk karena masih training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection')
BRANCH='agent/af2-igem-paired-confirmation'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    result=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src'))
os.chdir(REPO)

In [ ]:
import torch
from coffee_detector.drive_project import resolve_drive_project_root, require_project_artifact
assert torch.cuda.is_available(), 'Aktifkan T4 GPU'
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=('benchmarks/faruq-v3-synthetic-density-v1/setup_summary.json',))
SETUP=require_project_artifact(PROJECT_ROOT,'benchmarks/faruq-v3-synthetic-density-v1/setup_summary.json')
OUTPUT_ROOT=PROJECT_ROOT/'experiments/faruq-v3-retained-synthetic-density-v1'
print('GPU    :',torch.cuda.get_device_name(0))
print('PROJECT:',PROJECT_ROOT)
print('SETUP  :',SETUP)
print('OUTPUT :',OUTPUT_ROOT)

In [ ]:
MODELS=['D0FT','AF1','AF2','IGEM1','STB1','SAF1','LPS1','CPE0','CPE7','ACMC1','GEO1']
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_retained_synthetic_density','--setup-summary',str(SETUP),'--project-root',str(PROJECT_ROOT),'--output-root',str(OUTPUT_ROOT),'--checkout-root','/content/retained-model-branches','--models',*MODELS,'--device','0']
LOG=OUTPUT_ROOT/'run.log'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
print('MENJALANKAN retained-only synthetic benchmark')
process=subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with LOG.open('a',encoding='utf-8') as log:
    for line in process.stdout:
        print(line,end='',flush=True); log.write(line); log.flush()
code=process.wait()
if code != 0:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:]))
    raise RuntimeError(f'Benchmark gagal: {code}; log={LOG}')

In [ ]:
import json, pandas as pd
from IPython.display import display
SUMMARY=OUTPUT_ROOT/'retained_synthetic_density_summary.json'
result=json.loads(SUMMARY.read_text())
ranking=pd.DataFrame(result['ranking'])
display(ranking.style.format({'mean_macro_map50_95':'{:.2%}','mean_macro_delta_vs_d0ft':'{:+.2%}','minimum_macro_delta_vs_d0ft':'{:+.2%}'}))
rows=pd.DataFrame(result['rows'])
display(rows.pivot(index='model',columns='condition',values='delta_macro_map50_95_vs_d0ft').style.format('{:+.2%}'))
print('TRAINING:',result['training_executed'],'| TEST:',result['test_images_accessed'])
print('SUMMARY:',SUMMARY)